<a href="https://colab.research.google.com/github/ej070829/Halla/blob/main/Untitled10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2
import pytesseract
import re

img = cv2.imread("tyl.png")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 가장 큰 영역만 사용
mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)[1]
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, 8)

best_crop = None
max_area = 0

for i in range(1, num_labels):
    x, y, w, h, area = stats[i]
    if area > max_area:
        max_area = area
        best_crop = gray[y:y+h, x:x+w]

# 확대
up = cv2.resize(best_crop, None, fx=8, fy=8, interpolation=cv2.INTER_CUBIC)

# 대비 향상
enhanced = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8)).apply(up)

# 1. 블랙햇
kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (8, 8))
blackhat = cv2.morphologyEx(enhanced, cv2.MORPH_BLACKHAT, kernel)

# 2. 반전 이진화
_, binary_inv = cv2.threshold(enhanced, 140, 255, cv2.THRESH_BINARY_INV)

config = r'--oem 3 --psm 8 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ'

def clean_text(text):
    text = text.upper()
    text = re.sub(r'[^A-Z]', '', text)
    return text

results = {
    "enhanced": clean_text(pytesseract.image_to_string(enhanced, config=config)),
    "blackhat": clean_text(pytesseract.image_to_string(blackhat, config=config)),
    "binary_inv": clean_text(pytesseract.image_to_string(binary_inv, config=config)),
}
text=results["enhanced"]
print(results["enhanced"])